In [ ]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# ─────────────────────────────────────────
# 1. Definir W (2x3) aleatoria sin restricciones
# ─────────────────────────────────────────
np.random.seed(7)

W = np.random.randn(2, 3)

print("W (2x3):")
print(np.round(W, 4))
print("\nWWt (no necesariamente I):")
print(np.round(W @ W.T, 4))

# ─────────────────────────────────────────
# 2. SVD de W
# ─────────────────────────────────────────
U, S, Vt = np.linalg.svd(W)
print("\nValores singulares:", np.round(S, 4))
print("Vectores singulares derechos (filas de Vt):")
print(np.round(Vt, 4))

# ─────────────────────────────────────────
# 3. Generar nube de puntos x en R^3
# ─────────────────────────────────────────
N = 400
x_cloud = np.random.randn(3, N)

# ─────────────────────────────────────────
# 4. Calcular h = Wx en R^2
# ─────────────────────────────────────────
h_cloud = W @ x_cloud

# ─────────────────────────────────────────
# 5. Reconstrucción x' = Wt h
# ─────────────────────────────────────────
x_rec = W.T @ h_cloud  # (3, N)
error = np.linalg.norm(x_cloud - x_rec, axis=0)
print(f"\nError medio de reconstrucción: {error.mean():.4f}")

# ─────────────────────────────────────────
# 6. WtW (la proyección ortogonal)
# ─────────────────────────────────────────
WtW = W.T @ W  # 3x3

# Autovalores y autovectores de WtW
eigenvalues, eigenvectors = np.linalg.eigh(WtW)
# eigh devuelve en orden ascendente, invertimos
eigenvalues = eigenvalues[::-1]
eigenvectors = eigenvectors[:, ::-1]

print("\nWtW (3x3):")
print(np.round(WtW, 4))
print("\nAutovalores de WtW:", np.round(eigenvalues, 4))
print("(deben ser σ₁², σ₂², 0 =", np.round(S**2, 4), ", 0)")

# ─────────────────────────────────────────
# 7. Graficar — 2 filas x 2 columnas
# ─────────────────────────────────────────
plt.close('all')
# ── Fig 1: nube x en R^3 ──
fig1 = plt.figure(figsize=(7, 6))
ax1 = fig1.add_subplot(111, projection='3d')
ax1.scatter(*x_cloud, c='steelblue', s=8, alpha=0.3)
scale = 2.5
colors_v = ['crimson', 'royalblue', 'gray']
labels_v = [f'$v_1$ (σ={S[0]:.2f})', f'$v_2$ (σ={S[1]:.2f})', '$v_3$ (núcleo)']
for i in range(3):
    v = Vt[i]
    ax1.quiver(0,0,0, scale*v[0], scale*v[1], scale*v[2],
               color=colors_v[i], linewidth=2.2,
               arrow_length_ratio=0.15, label=labels_v[i],
               linestyle='-' if i < 2 else '--')
ax1.set_title("Nube $x$ en $\\mathbb{R}^3$\n(vectores singulares derechos de $W$)", fontsize=12)
ax1.set_xlabel("x₁"); ax1.set_ylabel("x₂"); ax1.set_zlabel("x₃")
ax1.legend(fontsize=9, loc='upper left')
ax1.set_xlim(-3,3); ax1.set_ylim(-3,3); ax1.set_zlim(-3,3)
fig1.tight_layout()
#fig1.savefig("/mnt/user-data/outputs/fig1_nube_x.png", dpi=150, bbox_inches='tight')

# ── Fig 2: nube h en R^2 ──
fig2, ax2 = plt.subplots(figsize=(6, 6))
ax2.scatter(*h_cloud, c='darkorange', s=8, alpha=0.9)
for i in range(2):
    u = U[:, i]
    ax2.annotate('', xy=(S[i]*u[0], S[i]*u[1]), xytext=(0,0),
                 arrowprops=dict(arrowstyle='->', color=['crimson','royalblue'][i], lw=2.5))
    ax2.text(S[i]*u[0]*1.18, S[i]*u[1]*1.18,
             f'$u_{i+1}$ (σ={S[i]:.2f})',
             color=['crimson','royalblue'][i], fontsize=10, ha='center')
ax2.set_title("Nube $h = Wx$ en $\\mathbb{R}^2$\n(vectores singulares izquierdos de $W$)", fontsize=12)
ax2.set_xlabel("h₁"); ax2.set_ylabel("h₂")
ax2.axhline(0, color='k', lw=0.5); ax2.axvline(0, color='k', lw=0.5)
ax2.set_aspect('equal'); ax2.grid(True, alpha=0.3)
lim = max(abs(h_cloud).max() * 1.2, 1)
ax2.set_xlim(-lim, lim); ax2.set_ylim(-lim, lim)
fig2.tight_layout()
#fig2.savefig("/mnt/user-data/outputs/fig2_nube_h.png", dpi=150, bbox_inches='tight')

# ── Fig 3: x vs x' ──
fig3, ax3 = plt.subplots(figsize=(6, 6))
ax3.scatter(x_cloud[0], x_rec[0], s=8, alpha=0.9, color='darkgreen', label='dim 1')
#ax3.scatter(x_cloud[1], x_rec[1], s=8, alpha=0.9, color='blue',   label='dim 2')
#ax3.scatter(x_cloud[2], x_rec[2], s=8, alpha=0.9, color='red',         label='dim 3')
lim2 = 3.5
ax3.plot([-lim2, lim2], [-lim2, lim2], 'k--', lw=1, label='recuperación perfecta')
ax3.set_title(f"$x$ vs $x' = W^T h$\nerror medio de reconstrucción = {error.mean():.2f}", fontsize=12)
ax3.set_xlabel("$x$"); ax3.set_ylabel("$x'$")
ax3.legend(fontsize=9); ax3.grid(True, alpha=0.3)
ax3.set_xlim(-lim2, lim2); ax3.set_ylim(-lim2, lim2)
ax3.set_aspect('equal')
fig3.tight_layout()
#fig3.savefig("/mnt/user-data/outputs/fig3_reconstruccion.png", dpi=150, bbox_inches='tight')
 
# ── Fig 4: WtW heatmap ──
fig4, ax4 = plt.subplots(figsize=(6, 5))
im = ax4.imshow(WtW, cmap='RdBu_r', vmin=-abs(WtW).max(), vmax=abs(WtW).max())
plt.colorbar(im, ax=ax4, shrink=0.8)
for i in range(3):
    for j in range(3):
        ax4.text(j, i, f'{WtW[i,j]:.2f}',
                 ha='center', va='center', fontsize=12,
                 color='white' if abs(WtW[i,j]) > abs(WtW).max()*0.5 else 'black',
                 fontweight='bold')
ax4.set_xticks([0,1,2]); ax4.set_xticklabels(['$x_1$','$x_2$','$x_3$'], fontsize=12)
ax4.set_yticks([0,1,2]); ax4.set_yticklabels(['$x_1$','$x_2$','$x_3$'], fontsize=12)
ax4.set_title("$W^TW$ — proyección ortogonal sobre espacio fila de $W$\n"
              f"Autovalores: σ₁²={eigenvalues[0]:.2f},  σ₂²={eigenvalues[1]:.2f},  0≈{eigenvalues[2]:.2e}",
              fontsize=11)
center = np.array([1.0, 1.0])
scale_eig = 0.85
ev_colors = ['crimson', 'royalblue', 'gray']
ev_labels = [f'$v_1$ (λ={eigenvalues[0]:.2f})',
             f'$v_2$ (λ={eigenvalues[1]:.2f})',
             f'$v_3$ (núcleo)']
for i in range(3):
    ev = eigenvectors[:, i]
    dx, dy = scale_eig * ev[0], scale_eig * ev[1]
    ax4.annotate('', xy=(center[0]+dx, center[1]+dy),
                 xytext=(center[0]-dx, center[1]-dy),
                 arrowprops=dict(arrowstyle='->', color=ev_colors[i], lw=2))
    ax4.text(center[0]+dx*1.3, center[1]+dy*1.3, ev_labels[i],
             color=ev_colors[i], fontsize=8, ha='center',
             bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.8, ec='none'))
fig4.tight_layout()

W (2x3):
[[ 1.6905 -0.4659  0.0328]
 [ 0.4075 -0.7889  0.0021]]

WWt (no necesariamente I):
[[3.0761 1.0566]
 [1.0566 0.7885]]

Valores singulares: [1.868  0.6125]
Vectores singulares derechos (filas de Vt):
[[ 0.9223 -0.3862  0.0168]
 [-0.3859 -0.9224 -0.0164]
 [-0.0218 -0.0086  0.9997]]

Error medio de reconstrucción: 2.3182

WtW (3x3):
[[ 3.0239e+00 -1.1092e+00  5.6300e-02]
 [-1.1092e+00  8.3950e-01 -1.6900e-02]
 [ 5.6300e-02 -1.6900e-02  1.1000e-03]]

Autovalores de WtW: [3.4894 0.3751 0.    ]
(deben ser σ₁², σ₂², 0 = [3.4894 0.3751] , 0)


In [28]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)
W = np.random.randn(2, 3)
U, S, Vt = np.linalg.svd(W)
WtW = W.T @ W

eigenvalues, eigenvectors = np.linalg.eigh(WtW)
eigenvalues = eigenvalues[::-1]
eigenvectors = eigenvectors[:, ::-1]

N = 400
x_cloud = np.random.randn(3, N)
x_proj = WtW @ x_cloud  # proyección de cada punto

fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(111, projection='3d')

# nube original (tenue) y nube proyectada
ax.scatter(*x_cloud, c='steelblue', s=8, alpha=0.15, label='$x$ original')
ax.scatter(*x_proj,  c='darkorange', s=8, alpha=0.35, label='$W^TWx$ (proyectado)')

# segmentos x → WtWx para algunos puntos
idx = np.random.choice(N, 60, replace=False)
for i in idx:
    ax.plot([x_cloud[0,i], x_proj[0,i]],
            [x_cloud[1,i], x_proj[1,i]],
            [x_cloud[2,i], x_proj[2,i]],
            color='gray', lw=0.6, alpha=0.5)

# autovectores escalados por su autovalor
scale = 2.5
colors  = ['crimson', 'royalblue', 'gray']
lstyles = ['-', '-', '--']
labels  = [f'$v_1$  (λ=σ₁²={eigenvalues[0]:.2f})',
           f'$v_2$  (λ=σ₂²={eigenvalues[1]:.2f})',
           f'$v_3$  (λ=0, núcleo)']

for i in range(3):
    v = eigenvectors[:, i]
    ax.quiver(0,0,0, scale*v[0], scale*v[1], scale*v[2],
              color=colors[i], linewidth=2.5,
              arrow_length_ratio=0.12,
              linestyle=lstyles[i],
              label=labels[i])

ax.set_title("Autovectores de $W^TW$ en $\\mathbb{R}^3$\n"
             "Azul/rojo: espacio fila de $W$  |  Gris: núcleo de $W$",
             fontsize=11)
ax.set_xlabel("x₁"); ax.set_ylabel("x₂"); ax.set_zlabel("x₃")
ax.legend(fontsize=9, loc='upper left')
ax.set_xlim(-3,3); ax.set_ylim(-3,3); ax.set_zlim(-3,3)

fig.tight_layout()
fig.savefig("/mnt/user-data/outputs/fig4_WtW_v2.png", dpi=150, bbox_inches='tight')
print("Guardado.")

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/user-data/outputs/fig4_WtW_v2.png'

In [29]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)
W = np.random.randn(2, 3)
U, S, Vt = np.linalg.svd(W)
WtW = W.T @ W

N = 400
x_cloud = np.random.randn(3, N)
x_proj  = WtW @ x_cloud  # WtWx, vive en el espacio fila de W

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Opción B: $x$ vs $W^TWx$ — dimensión a dimensión", fontsize=13, fontweight='bold')

dim_colors = ['mediumseagreen', 'mediumpurple', 'salmon']
dim_labels = ['dim 1 ($x_1$)', 'dim 2 ($x_2$)', 'dim 3 ($x_3$)']
lim = 3.5

for d in range(3):
    ax = axes[d]
    ax.scatter(x_cloud[d], x_proj[d], s=10, alpha=0.35, color=dim_colors[d])
    ax.plot([-lim, lim], [-lim, lim], 'k--', lw=1, label='sin cambio')

    # correlación como medida de cuánto se preserva
    corr = np.corrcoef(x_cloud[d], x_proj[d])[0,1]
    ax.set_title(f"{dim_labels[d]}\ncorrelación = {corr:.3f}", fontsize=11)
    ax.set_xlabel(f"$x_{d+1}$"); ax.set_ylabel(f"$(W^TW x)_{d+1}$")
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

fig.tight_layout()
